### Introduction to Machine Learning and Data Mining
# Randomized Cross-Validation with Pipelines — minimal example

**Goal:** tune a Random Forest with `RandomizedSearchCV`, using a `Pipeline` so that preprocessing is done *correctly* inside cross-validation.

**Data:** the Titanic passengers (1309 rows). Predict whether a passenger **survived**.
It is a good example because the data is *messy*: there are missing values and a text column.

In [33]:
import pandas as pd
from scipy.stats import randint

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

## 1. Data

We keep 4 predictors. `age` and `fare` are numbers with **missing values**; `sex` and `embarked` are **text categories**.
A Random Forest in scikit-learn needs numbers, so we cannot feed it this table directly.

In [34]:
titanic = fetch_openml("titanic", version=1, as_frame=True).frame

num_cols = ["age", "fare"]
cat_cols = ["sex", "embarked"]

X = titanic[num_cols + cat_cols]
y = titanic["survived"]

X.head()

,age,fare,sex,embarked
0,29.0000,211.3375,female,S
1,0.9167,151.5500,male,S
2,2.0000,151.5500,female,S
3,30.0000,151.5500,male,S
4,25.0000,151.5500,female,S


In [35]:
X.isna().sum()

age         263
fare          1
sex           0
embarked      2
dtype: int64

### Train / test split

The test set is locked away until the very end. Everything else (preprocessing *and* tuning) uses the training set only.

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0
)
print(X_train.shape, X_test.shape)

(1047, 4) (262, 4)


## 2. The problem: preprocessing + cross-validation

We need to (1) **fill the missing values** and (2) **turn text into numbers** (one-hot encoding).
Both steps *learn something from the data* (e.g. "the mean age is 29.9").

The tempting way:

```python
X_filled = imputer.fit_transform(X_train)        # learns the mean age from ALL training rows
cross_val_score(forest, X_filled, y_train)       # ...then cross-validates
```

This is **wrong**. In each CV round, some rows are supposed to be "unseen" validation data, but the imputer **already used them** to compute the mean age. Information from the validation rows leaked into the training step, so the CV score is too optimistic.

**Correct rule:** in every CV round, preprocessing must be learned **from the training folds only**, and then merely *applied* to the validation fold.
Doing that by hand is tedious and easy to forget. A **`Pipeline`** does it automatically.

## 3. Build the pipeline

A pipeline is a chain of steps that behaves like **one single model**:

```
raw table ──► [ prep: fill NaNs + one-hot ] ──► [ model: RandomForest ] ──► prediction
```

* `ColumnTransformer` applies a different preprocessing to each group of columns.
* `Pipeline` chains the preprocessing and the model.

In [5]:
prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="mean"),                 num_cols),   # fill missing numbers
    ("cat", OneHotEncoder(handle_unknown="ignore"),         cat_cols),   # text -> 0/1 columns
])

pipe = Pipeline([
    ("prep",  prep),
    ("model", RandomForestClassifier(random_state=0)),
])

pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output 

Now `pipe` accepts the **raw** table directly. `fit` learns the preprocessing *and* the forest; `predict` applies both.

In [6]:
pipe.fit(X_train, y_train)
print("Train accuracy:", round(pipe.score(X_train, y_train), 3))

Train accuracy: 0.968


## 4. Tuning the pipeline with randomized cross-validation

The forest reaches 0.968 accuracy on the **training** data, but it was *fit* on those very passengers, so this number says little about new ones. And we never chose its settings, such as the number of trees or the maximum depth: these **hyperparameters** are decided *before* training (unlike parameters, which the model learns), and different choices give different models.

Both problems are solved by **cross-validation (CV)**: split the training set into 5 folds, and 5 times *train on 4 folds and validate on the 5th*. The mean validation score is an honest estimate of performance for one configuration. Because `pipe` is a single object, each CV round clones the **whole pipeline** and refits the imputer, the encoder *and* the forest on the 4 training folds only, so there is no leakage.

To tune, we simply do this for many candidate configurations and keep the best one. Trying every combination (grid search) becomes very expensive, so `RandomizedSearchCV` tries only `n_iter` combinations drawn at random:

```
repeat n_iter times:
    1. draw a random configuration (e.g. n_estimators=217, max_depth=5, ...)
    2. cross-validate the pipeline with it
    3. record the mean CV score
keep the best configuration and refit it on the whole training set
```

Hyperparameter names inside a pipeline follow the pattern **`step__parameter`** (double underscore), and they can go as deep as needed:

| Name | Meaning |
|---|---|
| `model__n_estimators` | number of trees (a parameter of the *model* step) |
| `model__max_depth` | maximum depth of each tree |
| `model__min_samples_split` | minimum samples needed to split a node |
| `prep__num__strategy` | how to fill missing numbers: *prep* step → *num* transformer → `strategy` |

Notice that we tune a **preprocessing choice together with the model's hyperparameters** — only possible because both live in the same pipeline.

In [7]:
param_distributions = {
    "model__n_estimators":      randint(50, 300),        # any integer in [50, 300)
    "model__max_depth":         randint(2, 12),
    "model__min_samples_split": randint(2, 20),
    "prep__num__strategy":      ["mean", "median"],      # a list = pick one at random
}

search = RandomizedSearchCV(
    pipe,                          # the whole pipeline, not just the forest
    param_distributions,
    n_iter=20,                     # budget: 20 random combinations
    cv=5,                          # each combination is 5-fold cross-validated
    scoring="f1_macro",
    random_state=0,                # reproducible sampling
    n_jobs=-1,                     # use all CPU cores
)

search.fit(X_train, y_train);      # training data only!

In [8]:
print("Best CV macro-F1:", round(search.best_score_, 3))
print("Best parameters :")
print(pd.Series(search.best_params_).to_string())

Best CV macro-F1: 0.767
Best parameters :
model__max_depth               5
model__min_samples_split      16
model__n_estimators          217
prep__num__strategy         mean


`cv_results_` has one row per sampled combination. Here are the 5 best:

In [9]:
results = pd.DataFrame(search.cv_results_)
results.columns = [c.replace("param_", "") for c in results.columns]      # shorter names
cols = ["model__n_estimators", "model__max_depth", "model__min_samples_split",
        "prep__num__strategy", "mean_test_score"]
results.sort_values("mean_test_score", ascending=False)[cols].head(5).round(3).reset_index(drop=True)

,model__n_estimators,model__max_depth,model__min_samples_split,prep__num__strategy,mean_test_score
0,217,5,16,mean,0.767
1,192,4,2,median,0.766
2,164,5,13,median,0.765
3,215,10,19,median,0.764
4,137,7,6,mean,0.764


## 5. Final check on the test set (once)

`search.best_estimator_` is the best pipeline, already refit on the full training set. We pass it the **raw** test table: no manual imputing, no manual encoding, no risk of forgetting a step.

In [10]:
y_pred = search.predict(X_test)

print("Test accuracy :", round(accuracy_score(y_test, y_pred), 3))
print("Test macro-F1 :", round(f1_score(y_test, y_pred, average="macro"), 3))

Test accuracy : 0.786
Test macro-F1 : 0.77
